# 1. Setup

Import libraries and the create the output directory.

In [1]:
from pathlib import Path

import polars as pl
from pyscbwrapper import SCB

root = Path.cwd().resolve()

data_dir = root/ "data"
data_dir.mkdir(parents=True, exist_ok=True)

# 2. SCB Table Configuration

Define the table mapping and choose the target table ID



In [2]:
TABLES={
    "yr_rg_14_to_18": ("en", "AM", "AM0208", "AM0208M", "YREG60"),
    "yr_rg_19_to_21": ("en", "AM", "AM0208", "AM0208M", "YREG60N"),
    "yr_rg_20_to_24": ("en", "AM", "AM0208", "AM0208M", "YREG60BAS"),


}

tab_id = "yr_rg_20_to_24"

# 3. Initialize the SCB Client

Creating the API client for the selected SCB table

In [3]:
scb = SCB(*TABLES[tab_id])

In [4]:
#scb.info()

# 4. Load Variable Metadata

Read available dimensions and extract keys/values used to build the query.

In [5]:
var_ = scb.get_variables()

In [6]:
occ_key = next((k for k in var_ if "occupation" in k.lower()), None)

if occ_key:
    occupations = var_[occ_key]

occupations_key = "".join(occ_key.split()) # this is useful, it omits the whitespaces in the query



sex_key = next(k for k in var_ if "sex" in k.lower())
sex = var_[sex_key][:2]

region_key = next(k for k in var_ if "region" in k.lower())
regions = var_[region_key]
counties_only = [r for r in regions if r.lower().endswith("county")]


observations_key = next(k for k in var_ if "observations" in k.lower())
observations = var_[observations_key][0]

# 5. Build and Run Query

Set the dimension selections and fetch the data from SCB.

In [7]:
scb.set_query(
    **{
        occupations_key:occupations,
        sex_key:sex,
        region_key: counties_only,
        observations_key: observations,
    },
)

In [8]:
scb_data = scb.get_data()
scb_fetch = scb_data["data"]

# 6. Create Code Mappings 

Map SCB codes to readable occupation, sex and county labels.

In [9]:
codes = scb.get_query()["query"][0]["selection"]["values"]
occ_dict = dict(zip(codes, occupations, strict=False))


sex_codes = scb.get_query()["query"][1]["selection"]["values"]
sex_dict = dict(zip(sex_codes, sex, strict=False))

county_codes = scb.get_query()["query"][2]["selection"]["values"]
county_dict = dict(zip(county_codes, counties_only, strict=False))


# 7. Transform Response to DataFrame

Normalize API payload, clean records, and cast data types.


In [10]:
df = (
    pl.DataFrame(scb_fetch)
    .with_columns(
        # Step 1: Extract the raw data from the lists
        county_code = pl.col("key").list.get(0),
        code_4 = pl.col("key").list.get(1),
        sex_code = pl.col("key").list.get(2),
        year = pl.col("key").list.get(3),
        value = pl.col("values").list.get(0).cast(pl.Int64, strict=False),
    )
    .with_columns(
        # Step 2: Transform the extracted columns the stored dictionaries
        county = pl.col("county_code").replace(county_dict),
        occupation = pl.col("code_4").replace(occ_dict),
        sex = pl.col("sex_code").replace(sex_dict),
    )
    .filter(~pl.col("code_4").is_in(["0002", "0000"]))
    .select([
        "code_4", "occupation", "county_code",
        "county", "sex", "year", "value",
    ])
)

# 8. Preview Output

Check the transformed data

In [11]:
df.head(10)

code_4,occupation,county_code,county,sex,year,value
str,str,str,str,str,str,i64
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""men""","""2020""",745
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""men""","""2021""",794
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""men""","""2022""",814
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""men""","""2023""",909
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""men""","""2024""",768
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""women""","""2020""",80
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""women""","""2021""",80
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""women""","""2022""",106
"""0110""","""Commissioned armed forces offi…","""01""","""Stockholm county""","""women""","""2023""",110


# 9. Save Dataset 

Write the final dataset to Parquet in the `data` folder

In [12]:
df.write_parquet(data_dir/ "scb_yr_regions.parquet")